<div style="
  background: linear-gradient(145deg, #1a0b08, #2d1310);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #fff8f6;
  box-shadow: 0 6px 14px rgba(0,0,0,0.3);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #ff7b00, #ff0054, #9d0208);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  
  <b>06 $\rightarrow$ Production-Grade Offline vs. Online LLM Evaluations</b>
  <br>
  <span style="color:#ffb5a7; font-size: 18px;">(System Architecture, Telemetry Pipelines, and Closed-Loop Quality Control)</span>
</div>

---

# Table of Contents

1. [Architectural Overview: The Evaluation Lifecycle](#1-architectural-overview-the-evaluation-lifecycle)
   - 1.1 [Systemic Continuum: Staging vs. Production](#11-systemic-continuum-staging-vs-production)
   - 1.2 [The Fundamental Divergence: Correctness vs. Normality](#12-the-fundamental-divergence-correctness-vs-normality)

2. [Prerequisites](#2-prerequisites)
3. [Learning Objectives](#3-learning-objectives)
4. [Topic 1: Offline Evaluations (Pre-Deployment Staging)](#4-topic-1-offline-evaluations-pre-deployment-staging)
   - 4.1 [Overview](#41-overview)
   - 4.2 [Core Purposes: Pre-Release Gating, Version Benchmarking, and Regression Testing](#42-core-purposes-pre-release-gating-version-benchmarking-and-regression-testing)
   - 4.3 [Implementation Framework: Offline Automated CI/CD Regression Suite](#43-implementation-framework-offline-automated-cicd-regression-suite)
   - 4.4 [Code Walkthrough & Diagnostics](#44-code-walkthrough--diagnostics)
   - 4.5 [Best Practices & Common Mistakes](#45-best-practices--common-mistakes)
   - 4.6 [Key Takeaways](#46-key-takeaways)

5. [Topic 2: Production Risk Drivers & The Offline Paradigm Limit](#5-topic-2-production-risk-drivers--the-offline-paradigm-limit)
   - 5.1 [Overview](#51-overview)
   - 5.2 [Driver 1: Unanticipated Real-World Inputs](#52-driver-1-unanticipated-real-world-inputs)
   - 5.3 [Driver 2: Emergent & Scale-Dependent Failures](#53-driver-2-emergent--scale-dependent-failures)
   - 5.4 [Driver 3: Data and Concept Drift](#54-driver-3-data-and-concept-drift)
   - 5.5 [Best Practices & Common Mistakes](#55-best-practices--common-mistakes)
   - 5.6 [Key Takeaways](#56-key-takeaways)

6. [Topic 3: Online Evaluations (Post-Deployment Observability)](#6-topic-3-online-evaluations-post-deployment-observability)
   - 6.1 [Overview](#61-overview)
   - 6.2 [Structural Comparison Matrix: Offline vs. Online Evaluations](#62-structural-comparison-matrix-offline-vs-online-evaluations)
   - 6.3 [Evaluating Without Answer Keys: Statistical Baselines & Implicit Signals](#63-evaluating-without-answer-keys-statistical-baselines--implicit-signals)
   - 6.4 [Best Practices & Common Mistakes](#64-best-practices--common-mistakes)
   - 6.5 [Key Takeaways](#65-key-takeaways)

7. [Topic 4: Production Architecture of an Online Evaluation Pipeline](#7-topic-4-production-architecture-of-an-online-evaluation-pipeline)
   - 7.1 [Overview](#71-overview)
   - 7.2 [Step 1: Non-Blocking Telemetry Logging & Data Sanitization](#72-step-1-non-blocking-telemetry-logging--data-sanitization)
   - 7.3 [Step 2: Captured vs. Computed Signals](#73-step-2-captured-vs-computed-signals)
   - 7.4 [Step 3: Sampling Architectures (Random vs. Stratified/Targeted)](#74-step-3-sampling-architectures-random-vs-stratifiedtargeted)
   - 7.5 [Step 4: Time-Windowed Dashboards & Real-Time Alerting Systems](#75-step-4-time-windowed-dashboards--real-time-alerting-systems)
   - 7.6 [Implementation Framework: End-to-End Online Evaluator with PII Masking & Sampling](#76-implementation-framework-end-to-end-online-evaluator-with-pii-masking--sampling)
   - 7.7 [Code Walkthrough & Execution Output](#77-code-walkthrough--execution-output)
   - 7.8 [Best Practices & Common Mistakes](#78-best-practices--common-mistakes)
   - 7.9 [Key Takeaways](#79-key-takeaways)

8. [Topic 5: The Self-Improving Closed-Loop Architecture](#8-topic-5-the-self-improving-closed-loop-architecture)
   - 8.1 [Overview](#81-overview)
   - 8.2 [Production Failure Feedback Loops & Dataset Enrichment](#82-production-failure-feedback-loops--dataset-enrichment)
   - 8.3 [Key Takeaways](#83-key-takeaways)

9. [Cheat Sheet](#9-cheat-sheet)
10. [Glossary](#10-glossary)
11. [Final Summary](#11-final-summary)

---

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">1. Architectural Overview: The Evaluation Lifecycle</span>

<img src="../assets/nb_assets/nb0601.jpg" alt="nb0601.jpg" style="width:100%; max-width:700px; display:block; margin:auto;" />

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">1.2 The Fundamental Divergence: Correctness vs. Normality</span>

The engineering objectives of offline and online evaluation pipelines are distinct:

$$\text{Offline Evaluation Objective} \implies \text{Measure Correctness: } \mathcal{M}(Y_{\text{generated}}, Y_{\text{ground\_truth}}) \ge \tau$$

$$\text{Online Evaluation Objective} \implies \text{Measure Normality: } P(S_{\text{live\_window}}) \sim P(S_{\text{baseline\_distribution}})$$

* **Offline Evaluations measure Correctness**: Offline test suites verify whether system outputs match reference ground-truth requirements across factual, semantic, and structural dimensions.
* **Online Evaluations measure Normality**: Online systems monitor live streams to verify that system behavior, operational throughput, and metric score distributions remain within expected baseline bounds over time.

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  2. Prerequisites
</span>

Engineers studying this handbook should possess:

* **LLM Application Architecture**: Understanding of RAG pipelines, agentic orchestration, system prompts, and vector similarity search.
* **Data Engineering & Async Python**: Experience with Python asynchronous execution (`asyncio`), Pydantic validation, and structured logging.
* **Statistical Metric Formulation**: Familiarity with metric distributions, mean error calculations, and percentiles ($P_{50}, P_{90}, P_{99}$).
* **Production Observability**: Basic concepts of application telemetry, time-series dashboards, and alerting integrations.

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  3. Learning Objectives
</span>

By completing this course notebook, developers will be able to:

1. **Architect and Execute** automated offline evaluation pipelines integrated into CI/CD release gates.
2. **Identify and Mitigate** the three major production failure drivers: Unanticipated Inputs, Scale-Dependent Emergent Failures, and Concept Drift.
3. **Design** non-blocking, asynchronous telemetry logging infrastructures equipped with PII data masking protocols.
4. **Distinguish and Instrument** Captured Signals (latency, token costs) versus Computed Signals (faithfulness, toxicity, hallucination rates).
5. **Implement** Stratified Sampling strategies to optimize LLM-as-a-Judge evaluation costs over high-volume live traffic streams.
6. **Construct** a closed-loop evaluation system that feeds production failure logs back into offline golden datasets for regression testing.

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  4. Topic 1: Offline Evaluations (Pre-Deployment Staging)
</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.1 Overview</span>

An **Offline Evaluation** is a static, pre-deployment evaluation framework executed against a curated benchmark dataset ("Golden Dataset"). It measures whether system outputs satisfy quality criteria before code or configuration changes are pushed to live production servers.

### Offline Evaluation Function Matrix

| Function | Description & Operational Mechanism |
| --- | --- |
| **1. Pre-Release Gating (CI/CD Integration)** | Serves as a binary CI/CD release gate. Automatically blocks deployment if evaluation scores drop below predefined thresholds (e.g., Accuracy < 95%). |
| **2. Architectural & Version Benchmarking** | Benchmarks system variations (V1 vs V2) over identical ground-truth datasets (e.g., comparing Claude 3.5 Sonnet vs. GPT-4o, or testing re-ranker modules). |
| **3. Regression Testing** | Verifies that targeted prompt or component updates do not introduce unintended quality regressions across unrelated query categories. |

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.3 Implementation Framework: Offline Automated CI/CD Regression Suite</span>

The following Python script implements an offline regression test suite suitable for integration into automated CI/CD deployment pipelines. It executes candidate configurations over a version-controlled Golden Dataset, computes metric scores, and enforces hard release thresholds.

#### Prerequisites & Dependencies

```bash
pip install openai pydantic
```

In [1]:
# Offline Automated CI/CD Regression Test Suite

def run_cicd_suite(dataset, classifier_fn, gate_threshold=80.0):
    passed = 0
    for item in dataset:
        pred = classifier_fn(item["text"])
        if pred == item["label"]:
            passed += 1
            
    acc = (passed / len(dataset)) * 100.0
    gate_passed = acc >= gate_threshold
    
    print("=" * 60)
    print("OFFLINE CI/CD TEST SUITE RUN")
    print("=" * 60)
    print(f"Accuracy: {passed}/{len(dataset)} ({acc:.1f}%)")
    print(f"Build Gate Threshold: >= {gate_threshold}%")
    print(f"Build Result: {'[SUCCESS - BUILD PASSED]' if gate_passed else '[FAILURE - BUILD BLOCKED]'}")
    return gate_passed

golden_dataset = [
    {"text": "Cancel my account", "label": "Billing"},
    {"text": "Error 404 not found", "label": "Technical"},
    {"text": "Where is your office?", "label": "General"},
]

def mock_classifier(text):
    t = text.lower()
    if "cancel" in t: return "Billing"
    if "error" in t:  return "Technical"
    return "General"

run_cicd_suite(golden_dataset, mock_classifier)

OFFLINE CI/CD TEST SUITE RUN
Accuracy: 3/3 (100.0%)
Build Gate Threshold: >= 80.0%
Build Result: [SUCCESS - BUILD PASSED]


True

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.4 Code Walkthrough & Diagnostics</span>

1. **Golden Dataset Schema (`GroundTruthSample`)**: Enforces explicit structure across benchmark inputs, expected output categories, and mandatory factual target elements.
2. **Deterministic Model Execution (`T=0.0`)**: Sets temperature to zero to ensure stable, reproducible test scores across execution runs.
3. **CI/CD Release Gate Decision (`CICDReleaseReport`)**: Computes a weighted overall evaluation score and evaluates it against `min_release_threshold`, returning a boolean `release_approved` flag to gate production deployments.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.5 Best Practices & Common Mistakes</span>

#### Best Practices

* **Version Control Golden Datasets**: Maintain Golden Datasets in code repositories alongside application code, tracking changes over time.
* **Automate Gate Checks via GitHub Actions**: Trigger offline evaluation scripts automatically on every pull request targeting primary branches.

#### Common Mistakes

* **Overfitting to the Golden Dataset**: Optimizing prompts specifically to pass Golden Dataset queries while ignoring real-world input diversity.
* **Neglecting Target Updating**: Failing to update Golden Dataset reference values when underlying business rules or policies change.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">4.6 Key Takeaways</span>

* Offline evaluations act as automated quality gates, ensuring changes do not introduce regressions before code reaches production servers.
* Offline pipelines rely on static, ground-truth reference targets to measure explicit functional correctness.

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  5. Topic 2: Production Risk Drivers & The Offline Paradigm Limit
</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">5.1 Overview</span>

Passing offline staging tests is a prerequisite for deployment, but it does not guarantee production reliability. Production environments introduce operational complexities and input distributions that cannot be fully replicated in offline staging suites.

### Unanticipated Input Taxonomy

| Input Type | Operational Behavior & Risk |
| --- | --- |
| **Code-Switching & Hinglish** | Mixing languages mid-sentence (e.g., Hindi/English). Alters tokenization and semantic parser reliability. |
| **Ambiguous / Partial Inputs** | Fragmented user queries lacking necessary parameters. May trigger model hallucinations or incorrect assumptions. |
| **Emotional / Hostile Rants** | Inputs containing high emotional noise masking the actual underlying functional query. |
| **Adversarial Injections** | Prompt injection attempts designed to bypass system safety constraints and extract system instructions. |

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">5.3 Driver 2: Emergent & Scale-Dependent Failures</span>

Certain system failure modes manifest only under real-world scale and high concurrent load:

* **Concurrency & Latency Spikes**: High concurrent user traffic causes API queue delays and memory contention, driving Time-To-First-Token (TTFT) and total latency beyond acceptable thresholds.
* **Subtle Demographic Bias**: Systematic biases across specific user cohorts become detectable only when analyzing aggregated data across thousands of live production conversations.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">5.4 Driver 3: Data and Concept Drift</span>

Over time, the statistical distribution of production inputs shifts away from the static conditions used to construct the offline Golden Dataset:

<img src="../assets/nb_assets/nb0602.png" alt="nb0602.png" style="width:100%; max-width:700px; display:block; margin:auto;" />

* **Data Drift**: Shifts in the frequency or types of user queries over time.
* **Concept Drift**: Shifts in the relationship between input queries and correct outputs due to changing business logic, policies, or domain knowledge.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">5.5 Best Practices & Common Mistakes</span>

#### Best Practices

* **Implement Versioning for Knowledge Collections**: Re-evaluate and update Golden Datasets whenever underlying business documentation or RAG knowledge collections are updated.
* **Monitor Production Anomaly Clusters**: Group and analyze misclassified production inputs to identify emerging query categories missing from offline test suites.

#### Common Mistakes

* **Assuming High Staging Scores Guarantee Production Safety**: Believing that a $99\%$ offline test score eliminates the need for live production monitoring.
* **Ignoring Operational Performance Metrics**: Evaluating answer correctness during offline testing while ignoring live response latency and API throughput.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">5.6 Key Takeaways</span>

* Real-world production traffic exposes applications to unanticipated inputs, scale-dependent latency spikes, and distribution drift.
* Static offline datasets become obsolete over time as business rules and user query distributions evolve.

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  6. Topic 3: Online Evaluations (Post-Deployment Observability)
</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">6.1 Overview</span>

An **Online Evaluation** is a continuous observability framework that inspects live production traffic post-deployment. Operating directly on streaming user interactions, online evaluations monitor system health, detect operational anomalies, and track metrics without interfering with live user responses.

## Offline vs. Online Evaluation Matrix

| Architectural Dimension | Offline Evaluation (Staging) | Online Evaluation (Production) |
| --- | --- | --- |
| **Execution Phase** | Pre-Deployment (CI/CD Pipeline) | Post-Deployment (Live Stream) |
| **Dataset Type** | Fixed, Curated Golden Dataset | Unfiltered Streaming Traffic |
| **Ground-Truth Reference** | Mandatory Ground-Truth Reference Targets | No Reference Target Available |
| **Primary Benchmark** | Measures Functional Correctness | Measures Health & Normality |
| **Detected Anomalies** | Code & Prompt Regressions | Distribution Drift & Latency |
| **Evaluation Throughput** | High-Speed Batch (Small Test Volume) | Sampled Traffic Streams |
| **Primary Usage** | Release Gating & Version Benchmarking | Telemetry Alerts & Anomaly Detection |

### REFERENCE-FREE EVALUATION MECHANISMS

| **Category** | **Description** |
|--------------|-----------------|
| **1. COMPUTED REFERENCE-FREE METRICS** | - **Faithfulness:** Uses LLM judges to verify claims against retrieved context.<br>- **Toxicity & PII:** Audits outputs for safety policy violations. |
| **2. IMPLICIT & EXPLICIT USER SIGNALS** | - **Explicit Signals:** User thumbs-up / thumbs-down ratings.<br>- **Implicit Signals:** Session abandonments, support escalations, query rephrasing. |
| **3. STATISTICAL SCORE DISTRIBUTION BASELINES** | - Tracks metric score distributions over time (e.g., weekly average scores).<br>- Flags statistical shifts from historical baselines as potential anomalies. |

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">6.4 Best Practices & Common Mistakes</span>

#### Best Practices

* **Combine Implicit and Explicit Feedback**: Pair explicit user ratings (thumbs up/down) with implicit behavior signals (e.g., rapid query rephrasing or support email escalations).
* **Establish Moving Baseline Windows**: Use rolling 7-day or 30-day baseline windows to account for expected cyclical traffic variations.

#### Common Mistakes

* **Relying Solely on Explicit User Feedback**: Depending exclusively on thumbs-up/thumbs-down buttons, which are clicked by less than $2\%$ of active users.
* **Treating Individual Outliers as Systemic Failures**: Triggering critical alerts for single low-scoring interactions rather than tracking aggregate statistical trends.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">6.5 Key Takeaways</span>

* Online evaluations monitor live streaming traffic continuously to ensure system health and operational stability post-deployment.
* Operating without ground-truth reference targets, online pipelines leverage reference-free metrics, implicit user signals, and statistical score distributions to detect anomalies.

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  7. Topic 4: Production Architecture of an Online Evaluation Pipeline
</span>

<img src="../assets/nb_assets/nb0603.png" alt="nb0603.png" style="width:100%; max-width:700px; display:block; margin:auto;" />

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">7.2 Step 1: Non-Blocking Telemetry Logging & Data Sanitization</span>

Logging production telemetry must never increase end-user response latency. Logging engines execute asynchronously in worker threads or background tasks.

#### PII Sanitization Protocol

Before persisting conversation traces to long-term storage or observability platforms (e.g., LangSmith, Phoenix), logs pass through redaction filters to mask Personally Identifiable Information (PII):

```python
# PII Redaction Strategy
Raw User Input:  "My email is user@example.com and phone is 555-0199."
Sanitized Log:   "My email is [REDACTED_EMAIL] and phone is [REDACTED_PHONE]."
```

### CAPTURED vs. COMPUTED SIGNALS

| **Signal Category** | **Processing Mechanism** | **Examples** |
|---------------------|--------------------------|--------------|
| **1. Captured Signals**<br>*(Zero Model Cost)* | Extracted directly from runtime execution logs. No secondary model calls are required. | - Wall-Clock Latency (ms)<br>- Time-To-First-Token (TTFT)<br>- Prompt & Completion Token Counts<br>- Total Financial Cost ($)<br>- Explicit Thumbs Up/Down Ratings |
| **2. Computed Signals**<br>*(Requires Model)* | Evaluated asynchronously using secondary LLM judge calls. | - Contextual Faithfulness Score<br>- Toxicity & Harm Assessment<br>- Query Answer Relevance Score |

### SAMPLING STRATEGY COMPARISON

| **Strategy** | **Selection Mechanism** | **Pros / Cons** |
|--------------|--------------------------|-----------------|
| **Random Sampling** | Selects an arbitrary percentage (e.g., **5%**) of total traffic. | **Pros:** Simple to implement.<br>**Cons:** May miss rare edge-case failure events. |
| **Stratified / Targeted Sampling** | Categorizes traffic and samples disproportionately from high-risk segments (e.g., **100% of thumbs-down logs**, **2% of normal traffic**). | **Pros:** Highly cost-effective; focuses evaluator budget on likely failure cases.<br>**Cons:** Requires defining and maintaining sampling rules. |

<img src="../assets/nb_assets/nb0604.jpg" alt="nb0604.jpg" style="width:100%; max-width:700px; display:block; margin:auto;" />

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">7.6 Implementation Framework: End-to-End Online Evaluator with PII Masking & Sampling</span>

The following Python script implements an online telemetry evaluation pipeline featuring asynchronous PII masking, stratified sampling, reference-free faithfulness evaluation, and time-windowed threshold alerting.

#### Prerequisites & Dependencies

```bash
pip install openai pydantic
```

In [2]:
# Real-Time Online Telemetry & Monitoring Pipeline
import re

class OnlineTelemetry:
    def __init__(self, max_latency_ms=1000):
        self.max_latency_ms = max_latency_ms
        
    def sanitize(self, text):
        text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL_REDACTED]', text)
        return text

    def log(self, req_id, user_prompt, response, latency_ms):
        clean_prompt = self.sanitize(user_prompt)
        alerts = []
        if latency_ms > self.max_latency_ms:
            alerts.append("[ALERT: HIGH LATENCY]")
        if "[EMAIL_REDACTED]" in clean_prompt:
            alerts.append("[ALERT: PII DETECTED]")
            
        print(f"[{req_id}] Prompt: '{clean_prompt}' | Latency: {latency_ms}ms")
        if alerts:
            print(f"  Flags: {' '.join(alerts)}")

telemetry = OnlineTelemetry(max_latency_ms=800)
print("=" * 60)
print("ONLINE PRODUCTION TELEMETRY MONITORING")
print("=" * 60)

telemetry.log("REQ-01", "How do I reset password?", "Click forgot password.", 250)
telemetry.log("REQ-02", "Contact me at user@example.com", "Understood.", 400)
telemetry.log("REQ-03", "Generate long report", "Here is summary...", 1200)

ONLINE PRODUCTION TELEMETRY MONITORING
[REQ-01] Prompt: 'How do I reset password?' | Latency: 250ms
[REQ-02] Prompt: 'Contact me at [EMAIL_REDACTED]' | Latency: 400ms
  Flags: [ALERT: PII DETECTED]
[REQ-03] Prompt: 'Generate long report' | Latency: 1200ms
  Flags: [ALERT: HIGH LATENCY]


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">7.7 Code Walkthrough & Execution Output</span>

1. **PII Sanitization (`sanitize_pii_data`)**: Uses regular expressions to redact email addresses and phone numbers before persisting log records.
2. **Stratified Sampling Logic (`should_sample_trace`)**: Prioritizes traces receiving negative feedback (`thumbs_down`) or containing sensitive business keywords (`refund`, `billing`).
3. **Asynchronous Non-Blocking Execution (`evaluate_faithfulness_async`)**: Executes evaluation API calls asynchronously in background worker threads to avoid delaying user responses.
4. **Operational Alert Triggering**: Monitors captured latency signals in real time, firing alerts whenever latency exceeds target limits (`latency_ms > 1500.0`).

#### Expected Output

```text
--- STARTING ONLINE TELEMETRY EVALUATION PIPELINE SIMULATION ---

[EVALUATION TRACE] [TR-801]: Sampled for Eval | Faithfulness Score = 1.00
[OPERATIONAL ALERT] [TR-802]: Latency 1850.00ms exceeds threshold 1500.0ms!
[EVALUATION TRACE] [TR-802]: Sampled for Eval | Faithfulness Score = 1.00
```

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">7.8 Best Practices & Common Mistakes</span>

#### Best Practices

* **Implement Asynchronous Non-Blocking Workers**: Process telemetry logging and computed metrics in background queues (e.g., Celery, Redis) to protect user response times.
* **Enforce Strict PII Redaction**: Mask personal information at application boundaries before transmitting logs to external observability vendors.

#### Common Mistakes

* **Running Synchronous Model Evaluators in the Main Request Path**: Executing LLM-as-a-Judge evaluations synchronously inline with user API calls, doubling user response latency.
* **Evaluating 100% of High-Volume Production Traffic**: Running secondary model evaluators on all live queries, incurring unnecessary API costs.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">7.9 Key Takeaways</span>

* Production online pipelines require non-blocking telemetry logging, data sanitization, stratified sampling, and time-windowed alerting.
* Stratified sampling prioritizes high-risk query categories (e.g., negative feedback, email escalations), optimizing evaluation budget usage.

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  8. Topic 5: The Self-Improving Closed-Loop Architecture
</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">8.1 Overview</span>

Offline and online evaluation pipelines do not operate in isolation. They form a Self-Improving Closed-Loop Architecture where live production feedback continually enriches offline regression suites.

<img src="../assets/nb_assets/nb0605.png" alt="nb0605.png" style="width:100%; max-width:500px; display:block; margin:auto;" />

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #16A34A, #0F766E); padding: 7px 16px; border-radius: 6px; font-size: 16px; font-weight: 700; letter-spacing: 0.3px;">8.3 Key Takeaways</span>

* Online evaluations identify real-world failure modes and feed them back into offline Golden Datasets.
* Connecting online observability to offline test suites creates a self-improving engineering loop that increases application resilience over time.

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  9. Cheat Sheet
</span>

### Operational Reference & Architectural Rules

* **Offline vs. Online Operational Summary**:
  * **Offline**: Pre-deployment CI/CD testing; relies on static Golden Datasets with ground-truth answer keys; measures **Correctness**.
  * **Online**: Post-deployment observability; operates on live user streams using reference-free metrics; measures **Normality & Health**.

* **Sample Size & Cost Control Rule**:

$$\text{Sample Rate}_{\text{Targeted}} = \begin{cases} 100\% & \text{if User Feedback} = \text{'thumbs\_down'} \text{ or Query contains High-Risk Keywords} \\ 5\% - 10\% & \text{if Standard User Traffic} \end{cases}$$

* **PII Masking Regex Rules**:
  * **Email Pattern**: `r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'`
  * **Phone Pattern**: `r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b'`

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  10. Glossary
</span>

| Term | Definition |
| :--- | :--- |
| **Offline Evaluation** | Pre-deployment testing executed against a fixed benchmark dataset ("Golden Dataset") containing reference ground-truth answers. |
| **Online Evaluation** | Continuous post-deployment observability monitoring live streaming user traffic using reference-free metrics and telemetry. |
| **Golden Dataset** | A curated, version-controlled staging dataset containing representative prompts, optional contexts, and human-verified target outputs. |
| **Data Drift** | Shifts in the statistical distribution of production input queries over time. |
| **Concept Drift** | Shifts in the relationship between input queries and correct outputs due to changing business policies or external domain facts. |
| **PII Masking** | Redacting or sanitizing Personally Identifiable Information (e.g., emails, phone numbers, credit card numbers) before storing logs. |
| **Captured Signal** | Telemetry metrics extracted directly from runtime environments without calling secondary models (e.g., latency, token counts, cost). |
| **Computed Signal** | Evaluated metrics generated asynchronously by secondary judge models (e.g., faithfulness, toxicity, answer relevance). |
| **Stratified Sampling** | A sampling technique that selects higher proportions of data from high-risk categories (e.g., negative feedback, escalations) to control evaluation costs. |

##

<span style="
  display: inline-block;
  color: #fff;
  background: linear-gradient(135deg, #7209b7, #4cc9f0);
  padding: 12px 20px;
  border-radius: 12px;
  font-size: 24px;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(0,0,0,0.3);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
">
  11. Final Summary
</span>

Building production-grade AI software requires connecting **Offline Pre-Deployment Evaluations** and **Online Post-Deployment Observability** into a cohesive lifecycle.

By enforcing hard release gates during offline staging (measuring **Correctness** against Golden Datasets), capturing non-blocking telemetry and stratified metrics during online serving (measuring **Normality** over live streams), and closing the loop by feeding production failure logs back into offline test suites, engineering teams can build resilient, self-improving enterprise AI applications.